In [1]:
!pip install -q google-genai pydantic

In [11]:

# Task 4.1 - call Gemini for one note

import json
import time
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = "gemini-3.5-flash"

API_KEY = getpass("Paste your Gemini API key: ")
if not API_KEY.strip():
    raise ValueError("Please enter your API key.")

client = genai.Client(api_key=API_KEY)

ALLOWED = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = (
    "You help a chain of outpatient clinics understand why patients miss "
    "appointments. Read the note carefully and reply only with JSON."
)

api_calls = 0
last_call_time = 0
last_raw = ""


def craft_prompt(note_text):
    return f"""
Context: CareLoop Clinics wants to understand why patients miss appointments.
The note may be in English or Hinglish.

Role: You are a patient-experience analyst at the clinic.

Action: Identify the main reason for the missed appointment.
Choose exactly one root_cause from: {", ".join(ALLOWED)}.
If the reason is unclear, choose "other" rather than guessing.
Give confidence from 0 to 1, a summary of at most 25 words,
and one practical recommended_action.

Tone: Neutral and factual. Do not invent details.

Format: Return JSON with root_cause, confidence, summary,
and recommended_action.

Note: {note_text}
"""


def ask_gemini(prompt):
    global api_calls, last_call_time, last_raw

    # Wait 13 seconds between API calls
    wait = 13 - (time.time() - last_call_time)
    if wait > 0:
        time.sleep(wait)

    last_raw = ""
    api_calls += 1
    last_call_time = time.time()

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    last_raw = response.text
    return response.text


def analyse_note(note_text):
    raw = ask_gemini(craft_prompt(note_text))
    return json.loads(raw)


notes = pd.read_csv("/content/ecWvPsdV2FNmRZAa.csv")

# Use the first row for N001, without filtering by its ID
first_note = notes.iloc[0]
print("Note ID:", first_note["note_id"])

result = analyse_note(first_note["note_text"])
print(result)
print("API calls so far:", api_calls)

Paste your Gemini API key: ··········
Note ID: N001
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she did not receive a reminder message and forgot about the scheduled visit.', 'recommended_action': 'Verify patient contact details and notification settings, send automated SMS reminders, and rebook her for next week.'}
API calls so far: 1


In [12]:
# Task 4.2 - validate the first 10 notes

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def root_cause_allowed(insight):
    return insight.root_cause in ALLOWED


fields = [
    "root_cause",
    "confidence",
    "summary",
    "recommended_action"
]

rows = []

for _, note in notes.head(10).iterrows():
    row = {
        "note_id": note["note_id"],
        "appointment_id": note["appointment_id"]
    }

    note_text = note["note_text"]

    # Skip missing or blank notes without calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":
        row.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        rows.append(row)
        print(note["note_id"], "-> skipped blank note")
        continue

    raw = ""
    data = {}

    try:
        raw = ask_gemini(craft_prompt(str(note_text)))
        data = json.loads(raw)

        # Keep the fields as Gemini returned them
        for field in fields:
            row[field] = data.get(field, "")

        insight = FollowupInsight(**data)

        if root_cause_allowed(insight):
            row["validation_status"] = "valid"
        else:
            row["validation_status"] = "invalid"

    except Exception as e:
        row["validation_status"] = "invalid"
        print("Problem with", note["note_id"], ":", e)

        # Keep any fields that were returned before the error
        for field in fields:
            row.setdefault(field, data.get(field, ""))

    row["raw_response"] = raw
    rows.append(row)

    print(
        note["note_id"], "->",
        row.get("root_cause", ""),
        "|", row["validation_status"]
    )

results = pd.DataFrame(
    rows,
    columns=[
        "note_id", "appointment_id", "root_cause", "confidence",
        "summary", "recommended_action", "validation_status", "raw_response"
    ]
)

results.to_csv("notes_analysed.csv", index=False)

print("\nSaved notes_analysed.csv:", results.shape)
print("API calls so far:", api_calls)
print(results[["note_id", "root_cause", "validation_status"]])

N001 -> forgot_or_no_reminder | valid
N002 -> distance_or_transport | valid
N003 -> cost_or_fee | valid
N004 -> skipped blank note
N005 -> illness_or_emergency | valid
N006 -> work_or_schedule | valid
N007 -> forgot_or_no_reminder | valid
N008 -> distance_or_transport | valid
N009 -> other | valid
N010 -> work_or_schedule | valid

Saved notes_analysed.csv: (10, 8)
API calls so far: 10
  note_id             root_cause validation_status
0    N001  forgot_or_no_reminder             valid
1    N002  distance_or_transport             valid
2    N003            cost_or_fee             valid
3    N004                  other     skipped_blank
4    N005   illness_or_emergency             valid
5    N006       work_or_schedule             valid
6    N007  forgot_or_no_reminder             valid
7    N008  distance_or_transport             valid
8    N009                  other             valid
9    N010       work_or_schedule             valid


In [13]:
# Task 4.3(a) - test two bad replies

bad_replies = [
    {
        "root_cause": "cost_or_fee",
        "confidence": 1.7,
        "summary": "Patient found the fee too high.",
        "recommended_action": "Offer a payment option."
    },
    {
        "root_cause": "cost_or_fee",
        "confidence": 0.8,
        "summary": "Patient found the fee too high."
        # recommended_action is missing
    }
]

for bad in bad_replies:
    try:
        FollowupInsight(**bad)
        print("Reply passed validation.")
    except ValidationError as e:
        print("Caught validation error:")
        print(e)
        print()

Caught validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Caught validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ound the fee too high.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [14]:
# Task 4.3(b) - compare the two prompts on N007 and N009

def v1_prompt(note_text):
    return f"Classify this note: {note_text}"


comparison_rows = []

test_notes = notes[notes["note_id"].isin(["N007", "N009"])]

for _, note in test_notes.iterrows():
    for version, make_prompt in [
        ("V1", v1_prompt),
        ("V2", craft_prompt)
    ]:
        cause = ""
        status = "invalid"

        try:
            raw = ask_gemini(make_prompt(str(note["note_text"])))
            data = json.loads(raw)

            cause = data.get("root_cause", "")

            if cause in ALLOWED:
                status = "valid"

        except Exception as e:
            print("Problem with", note["note_id"], version, ":", e)

        comparison_rows.append({
            "note_id": note["note_id"],
            "prompt": version,
            "root_cause": cause,
            "validation_status": status
        })

comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))

print(
    "\nI would deploy V2 because it lists the allowed causes "
    "and tells the model to choose other when the note is unclear."
)

Problem with N007 V1 : 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


KeyboardInterrupt: 

In [15]:
# Task 4.3(c) - count API calls made so far

print("Total API calls in Tasks 4.1-4.3:", api_calls)

Total API calls in Tasks 4.1-4.3: 14


In [16]:
# Task 4.4 - merge the notes with appointment data

analysed = pd.read_csv("/content/notes_analysed.csv")
appointments = pd.read_csv("/content/appointments_clean.csv")

merged = analysed.merge(
    appointments,
    on="appointment_id",
    how="left"
)

print("Rows after merge:", len(merged))

cause_reminder = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"],
    margins=True
)

print(cause_reminder)

merged.to_csv("merged_notes_appointments.csv", index=False)
cause_reminder.to_csv("root_cause_vs_reminder.csv")

Rows after merge: 10
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10
